# معمل التجربة: قاعدة البيانات السحابية DataStax AstraDB

قاعدة بيانات AstraDB بتوفر محرك متجهات سحابي مبني على Apache Cassandra، مناسب للشركات اللي عندها مليارات المتجهات ومحتاجة توزيع جغرافي متعدد السحاب.

في المعمل ده:
1. هنفحص بيانات الاتصال السحابية.
2. هنجرب الاستعلام في قاعدة البيانات السحابية أو المحاكاة الآمنة لو المفتاح مش متوفر.

### 1. فحص بيانات الاتصال وتجهيز البيئة
يلا نتأكد من إعدادات المفاتيح:

In [1]:
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
endpoint = os.getenv("ASTRA_DB_API_ENDPOINT")
token = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(f"هل مفتاح AstraDB متوفر؟ {bool(endpoint and token)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

هل مفتاح AstraDB متوفر؟ True


### 2. الاتصال والتجربة
هننشئ الفهرس السحابي أو المحاكي ونبحث فيه:

In [2]:
sample_docs = [
    Document(page_content="قاعدة AstraDB السحابية تدعم متجهات كاساندرا على نطاق واسع.", metadata={"cloud": True}),
    Document(page_content="توفر AstraDB واجهة برمجية متوافقة مع LangChain لسهولة التكامل.", metadata={"cloud": True}),
]

if endpoint and token and len(token) > 10:
    try:
        from langchain_astradb import AstraDBVectorStore
        store = AstraDBVectorStore(collection_name="playground_demo", embedding=embeddings, api_endpoint=endpoint, token=token)
        store.add_documents(sample_docs)
        print("تم الاتصال بنجاح بـ AstraDB السحابية!")
    except Exception as e:
        print(f"تعذر الاتصال بالسحابة ({e})، سنستخدم المحاكي المحلي.")
        from langchain_core.vectorstores import InMemoryVectorStore
        store = InMemoryVectorStore.from_documents(sample_docs, embeddings)
else:
    from langchain_core.vectorstores import InMemoryVectorStore
    store = InMemoryVectorStore.from_documents(sample_docs, embeddings)
    print("تم تفعيل المحاكي المحلي لـ AstraDB.")

results = store.similarity_search("ما هي مميزات AstraDB؟", k=1)
print(f"النتيجة المسترجعة: {results[0].page_content}")

تم الاتصال بنجاح بـ AstraDB السحابية!


النتيجة المسترجعة: توفر AstraDB واجهة برمجية متوافقة مع LangChain لسهولة التكامل.


### 3. جرب بنفسك!
تقدر تضيف بيانات وتجرب الاستعلام السحابي بنفسك.